In [ ]:
import os
import sys
from pathlib import Path
from typing import Tuple

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "PI_score_generator.py").exists():
            return candidate
    raise RuntimeError("Could not find project root containing PI_score_generator.py")


project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from PI_score_generator import score_all, _load_lexicons, _PATTERN_CACHE
from persuasion_runner import build_score_matrices as project_build_score_matrices
from persuasion_runner import run_expanded_lexicons

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
print(f"Project root: {project_root}")


In [ ]:
# Notebook setup is handled in the first cell. Keep this cell as a visual separator.


# Functions

In [ ]:
def build_score_matrices(
    df: pd.DataFrame,
    text_col: str = "argument",
    use_expanded_lexicons: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs score_all once per row and returns two DataFrames:
    1. df_subfeatures: individual metrics such as Sentiment.anger
    2. df_means: category-level means such as Sentiment.mean
    """
    return project_build_score_matrices(
        df,
        text_col=text_col,
        use_expanded_lexicons=use_expanded_lexicons,
    )


In [ ]:
# Newly added function to score a single argument and return both sub-feature and mean scores
def score_single_argument(
    text: str,
    use_expanded_lexicons: bool = False
):
    df = pd.DataFrame({"argument": [text]})
    df_sub, df_mean = build_score_matrices(
        df,
        use_expanded_lexicons=use_expanded_lexicons
    )
    return df_sub.iloc[0], df_mean.iloc[0]

In [ ]:
from persuasion_profile import calculate_dual_weighted_score, get_persuasion_report


# Test multiple arguments

In [ ]:
import pandas as pd
# Evidence-heavy + Authority + Specificity
sub, mean = score_single_argument(
    "According to the World Health Organization, over 75% of adults experience significant stress at least once per year. A recent 2023 study conducted by researchers at Harvard University found that individuals who engage in regular physical activity reduce their stress levels by approximately 30% within three months. These findings suggest that even small, consistent changes—such as walking 2 kilometers at the end of the day per day—can have measurable benefits for mental health.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

In [ ]:
sub, mean = score_single_argument(
    "According to the World Health Organization, over 75% of adults experience significant stress at least once per year. A recent 2023 study conducted by researchers at Harvard University found that individuals who engage in regular physical activity reduce their stress levels by approximately 30% within three months. These findings suggest that even small, consistent changes—such as walking 2 kilometers at the end of the day per day—can have measurable benefits for mental health.",
    use_expanded_lexicons=True
)

print(sub[sub > 0].sort_values(ascending=False))

In [ ]:
import pandas as pd

# Emotional + Urgency + Persuasion
sub, mean = score_single_argument(
    "This is not just a minor issue—it is a serious and immediate threat to everything we care about. If we fail to act now, the consequences could be devastating for future generations. Imagine a world where clean air is no longer guaranteed and safe water becomes a luxury. We cannot afford to wait. We must act now before it is too late.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

In [ ]:
original = ['tons', 'p-value', 'mph', 'ft', 'jpy', 'l', 'hz', 'coefficient', 'ppm', 'km', 'liter', 'significant', 'percent', 'pixels', 'ton', 'index', 'billion', 'thousand', 'in', 'oz', 'db', 'ratio', 'g', 'kg', 'cm', 'mi', 'mm', 'percentage', 'watts', 'mb', 'eur', 'currency', '%', 'lb', 'gbp', 'v', 'mg', 'usd', 'gb', 'litre', 'tb', 'dpi', 'm', 'points', 'million', 'ppb', 'ml', 'kph']
expanded = ['megawatt', 'calories', 'tons', 'lumen', 'kilowatt', 'p-value', 'mph', 'jpy', 'coefficient', 'ft', 'hz', 'l', 'liter', 'ppm', 'km', 'significant', 'quart', 'percent', 'pixels', 'ton', 'index', 'billion', 'thousand', 'hectare', 'nanometer', 'in', 'oz', 'db', 'acre', 'ratio', 'kg', 'g', 'teraflop', 'cm', 'mi', 'mm', 'percentage', 'watts', 'eur', 'mb', 'currency', 'joules', '%', 'lb', 'gbp', 'gigahertz', 'v', 'mg', 'usd', 'gb', 'litre', 'tb', 'points', 'dpi', 'm', 'kilobyte', 'terabyte', 'million', 'decibel', 'ppb', 'gallon', 'ml', 'kph']
print(len(original), len(expanded))

In [ ]:
# Politeness + Indirect Persuasion
sub, mean = score_single_argument(
    "Hi everyone, I just wanted to share a quick thought. Would you be open to considering a small change in how we approach this problem? It might be helpful to try a slightly different strategy, especially since we have already seen some limitations with the current approach. I would really appreciate your feedback on this, and I think together we can find a better solution.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

In [ ]:
# Specificity + Concrete Details
sub, mean = score_single_argument(
    "The package was delivered at 3:45 PM on Tuesday and left at the front door of the building on 5th Avenue. Inside the box, there were three items: a 2 kg steel tool, a set of four replacement parts, and a printed instruction manual. Each component was individually wrapped in plastic and labeled with a serial number.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

In [ ]:
# Rapport + Friendly Tone
sub, mean = score_single_argument(
    "Hey! Thanks so much for taking the time to read this. I really appreciate your support and everything you’ve contributed so far. It’s been great working together, and I’m excited about what we can accomplish next. Let’s keep building on this momentum!",
    use_expanded_lexicons=False
)

print(sub[sub > 0].sort_values(ascending=False))
# print(sub[sub==0])

In [ ]:
# Strong Directive / High Persuasive Force
sub, mean = score_single_argument(
    "You need to make this decision today. Delaying any further will only increase the risks and make the situation harder to control. Take action now, follow the recommended steps, and ensure that the problem is addressed immediately. This is the most effective way to prevent further damage.",
    use_expanded_lexicons=False
)

print(sub[sub > 0].sort_values(ascending=False))
# print(sub[sub==0])

In [ ]:
# Balanced Argument (good for sanity check)
sub, mean = score_single_argument(
    "There are both advantages and disadvantages to this approach. On the one hand, it can improve efficiency and reduce costs in the short term. On the other hand, it may introduce new risks that need to be carefully managed. Ultimately, the best decision will depend on how these trade-offs are evaluated in the specific context.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

In [ ]:
# Mixed Strategy (REALISTIC persuasive argument)
sub, mean = score_single_argument(
    "According to recent reports from the United Nations, global temperatures have already increased by more than 1.1 degrees Celsius since pre-industrial times. This is not just a statistic—it represents real and irreversible damage to ecosystems around the world. If we continue on this path, millions of people could face severe consequences, including food shortages and displacement. We have the tools and knowledge to change this trajectory, but we must act now. Even small actions, such as reducing daily energy consumption or supporting sustainable policies, can make a meaningful difference.",
    use_expanded_lexicons=False
)

print(sub[sub > 0].sort_values(ascending=False))

In [ ]:
# Wrong spelling
sub, mean = score_single_argument(
    "Thee ae bth advantages and disadvantages to this approach. On the one hand, it can improve efficiency and reduce costs in the short term. On the other hand, it may introduce new risks that need to be carefully managed. Ultimately, the best decision will depend on how these trade-offs are evaluated in the specific context.",
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
print(sub[sub==0])

# Single comparison

In [ ]:
text1 = "Banning single-use plastics is a necessary step if we want to meaningfully reduce environmental damage in our city. Every year, Americans generate over 35 million tons of plastic waste, and less than 10% is successfully recycled. Much of the rest ends up in landfills or waterways, where it breaks down into microplastics that contaminate drinking water and harm marine life. Cities like San Francisco and Seattle have already implemented similar bans and seen measurable reductions in plastic waste without harming local businesses. This policy is not about inconvenience—it’s about long-term public health and sustainability. By transitioning to reusable alternatives, we reduce pollution, protect ecosystems, and lower cleanup costs that taxpayers ultimately bear. If we act now, we can prevent further damage and set a precedent for responsible consumption."

text2 = "We should probably ban single-use plastics because they are bad for the environment. There is a lot of plastic everywhere and it doesn’t look good. People throw things away and it just piles up, which is not great. Other places have done bans too, so maybe we should as well. It might help reduce waste, and using reusable things could be better. Overall, banning plastics seems like a good idea and could improve things somewhat."

raw_scores_1, ukp_weighted_1 = get_persuasion_report(text1)
raw_scores_2, ukp_weighted_2 = get_persuasion_report(text2)

print(ukp_weighted_1['metadata']['sub_model']['prob'],
      ukp_weighted_1['metadata']['mean_model']['prob'])

print(ukp_weighted_2['metadata']['sub_model']['prob'],
      ukp_weighted_2['metadata']['mean_model']['prob'])

In [ ]:
import pandas as pd
import numpy as np

def compare_reports_by_category(raw_scores_1, ukp_weighted_1, raw_scores_2, ukp_weighted_2):
    rows = []

    # all categories appearing in either report
    categories = sorted(set(raw_scores_1.keys()) | set(raw_scores_2.keys()))

    for cat in categories:
        raw_cat_1 = raw_scores_1.get(cat, {})
        raw_cat_2 = raw_scores_2.get(cat, {})
        weighted_cat_1 = ukp_weighted_1.get(cat, {})
        weighted_cat_2 = ukp_weighted_2.get(cat, {})

        # all subfeatures appearing in either raw/weighted dict
        subkeys = sorted(
            set(raw_cat_1.keys()) | set(raw_cat_2.keys()) |
            set(weighted_cat_1.keys()) | set(weighted_cat_2.keys())
        )

        for subk in subkeys:
            rows.append({
                "category": cat,
                "subfeature": subk,
                "raw_text1": raw_cat_1.get(subk, np.nan),
                "raw_text2": raw_cat_2.get(subk, np.nan),
                "raw_diff_1_minus_2": (
                    raw_cat_1.get(subk, np.nan) - raw_cat_2.get(subk, np.nan)
                    if pd.notna(raw_cat_1.get(subk, np.nan)) and pd.notna(raw_cat_2.get(subk, np.nan))
                    else np.nan
                ),
                "weighted_text1": weighted_cat_1.get(subk, np.nan),
                "weighted_text2": weighted_cat_2.get(subk, np.nan),
                "weighted_diff_1_minus_2": (
                    weighted_cat_1.get(subk, np.nan) - weighted_cat_2.get(subk, np.nan)
                    if pd.notna(weighted_cat_1.get(subk, np.nan)) and pd.notna(weighted_cat_2.get(subk, np.nan))
                    else np.nan
                ),
            })

    df = pd.DataFrame(rows)
    return df

In [ ]:
df_compare = compare_reports_by_category(
    raw_scores_1, ukp_weighted_1,
    raw_scores_2, ukp_weighted_2
)

df_compare

# Lexicon validation

In [ ]:
def build_score_matrices_no_lexicon_specific(
    df: pd.DataFrame,
    text_col: str = "argument"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Runs score_all once per row and returns two DataFrames:
    1. df_subfeatures: Every individual metric (e.g., 'Sentiment.anger')
    2. df_means: Only the category-level means (e.g., 'Sentiment.mean')
    """
    # run_expanded_lexicons(use_expanded_lexicons)

    rows_sub = []
    rows_mean = []

    for txt in df[text_col].fillna(""):
        # Process the text once per row
        txt = txt.lower()
        scores = score_all(txt)
        
        flat_sub = {}
        flat_mean = {}

        for cat, vals in scores.items():
            if isinstance(vals, dict):
                for subk, v in vals.items():
                    if subk == "mean":
                        # Populate the Mean Matrix
                        flat_mean[f"{cat}.mean"] = v
                    else:
                        # Populate the Sub-feature Matrix
                        flat_sub[f"{cat}.{subk}"] = v
            # Handle overall_mean if it's a direct key in your score_all output
            elif cat == "Overall_mean":
                flat_mean["Overall_mean"] = vals

        rows_sub.append(flat_sub)
        rows_mean.append(flat_mean)

    # Convert to DataFrames and handle numeric conversion/NaNs
    df_subfeatures = pd.DataFrame(rows_sub, index=df.index).apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df_means = pd.DataFrame(rows_mean, index=df.index).apply(pd.to_numeric, errors="coerce").fillna(0.0)

    return df_subfeatures, df_means

# Newly added function to score a single argument and return both sub-feature and mean scores
def score_single_argument_no_lexicon_specific(
    text: str
):
    df = pd.DataFrame({"argument": [text]})
    df_sub, df_mean = build_score_matrices_no_lexicon_specific(
        df
    )
    return df_sub.iloc[0], df_mean.iloc[0]

In [ ]:
import json
import random

random.seed(42)

HALF_SAMPLE_DIR = project_root / "lexicons" / "lexicons_validation" / "half_samples"
SAMPLE_1_PATH = HALF_SAMPLE_DIR / "lexicons_sample_1.json"
SAMPLE_2_PATH = HALF_SAMPLE_DIR / "lexicons_sample_2.json"


def _split_value(v):
    """Randomly shuffle and then split into (first_half, second_half)."""
    if isinstance(v, list):
        temp_list = list(v)
        random.shuffle(temp_list)
        mid = len(temp_list) // 2
        return temp_list[:mid], temp_list[mid:]

    if isinstance(v, dict):
        first, second = {}, {}
        for subk, subv in v.items():
            temp_sublist = list(subv)
            random.shuffle(temp_sublist)
            mid = len(temp_sublist) // 2
            first[subk] = temp_sublist[:mid]
            second[subk] = temp_sublist[mid:]
        return first, second

    return v, v


def split_lexicons_expanded(
    src_path: Path = project_root / "helper_features" / "lexicons_expanded_LLM_audited.json",
    out_dir: Path = HALF_SAMPLE_DIR,
    out_name_1: str = "lexicons_sample_1.json",
    out_name_2: str = "lexicons_sample_2.json",
) -> None:
    """
    Splits each word list in the expanded LLM-audited lexicon into first and
    second halves. Dict-of-lists values are split per sub-key.
    """
    with open(src_path, encoding="utf-8") as f:
        data = json.load(f)

    first_half, second_half = {}, {}
    for key, words in data.items():
        first_half[key], second_half[key] = _split_value(words)

    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / out_name_1, "w", encoding="utf-8") as f:
        json.dump(first_half, f, indent=2, ensure_ascii=False)
    with open(out_dir / out_name_2, "w", encoding="utf-8") as f:
        json.dump(second_half, f, indent=2, ensure_ascii=False)

    print(f"Saved {out_name_1} and {out_name_2} to '{out_dir}/'")
    for key in list(data.keys())[:3]:
        v = data[key]
        n = len(v) if isinstance(v, list) else sum(len(sv) for sv in v.values())
        n1 = len(first_half[key]) if isinstance(first_half[key], list) else sum(len(sv) for sv in first_half[key].values())
        n2 = len(second_half[key]) if isinstance(second_half[key], list) else sum(len(sv) for sv in second_half[key].values())
        print(f"  {key}: {n} -> {n1} / {n2}")
    print(f"  ... ({len(data)} keys total)")


split_lexicons_expanded()


In [ ]:
from PI_score_generator import _PATTERN_CACHE


def score_with_sample(sample_path: Path, text: str):
    os.environ["PI_LEXICON_FILE"] = str(sample_path.resolve())
    _load_lexicons.cache_clear()
    _PATTERN_CACHE.clear()
    return score_single_argument_no_lexicon_specific(text)


test_text = (
    "According to the World Health Organization, over 75% of adults experience significant stress "
    "at least once per year. A recent 2023 study conducted by researchers at Harvard University found "
    "that individuals who engage in regular physical activity reduce their stress levels by approximately "
    "30% within three months. These findings suggest that even small, consistent changes, such as walking "
    "2 kilometers per day, can have measurable benefits for mental health."
)

sub1, mean1 = score_with_sample(SAMPLE_1_PATH, test_text)
sub2, mean2 = score_with_sample(SAMPLE_2_PATH, test_text)

df_sub_compare = pd.DataFrame({
    "sample_1": sub1,
    "sample_2": sub2,
    "diff (1-2)": sub1 - sub2,
}).sort_values("diff (1-2)", key=abs, ascending=False)

df_mean_compare = pd.DataFrame({
    "sample_1": mean1,
    "sample_2": mean2,
    "diff (1-2)": mean1 - mean2,
}).sort_values("diff (1-2)", key=abs, ascending=False)

print("=== Sub-feature comparison ===")
display(df_sub_compare)
print("\n=== Mean comparison ===")
display(df_mean_compare)


In [ ]:
import pandas as pd
# Evidence-heavy + Authority + Specificity
sub, mean = score_single_argument(
    test_text,
    use_expanded_lexicons=True
)

# print(sub[sub > 0].sort_values(ascending=False))
mean

In [ ]:
import pandas as pd
# Evidence-heavy + Authority + Specificity
sub, mean = score_single_argument(
    test_text,
    use_expanded_lexicons=False
)

# print(sub[sub > 0].sort_values(ascending=False))
mean

# Internal Consistency on UKP Dataset

Score every UKP argument (train + test) with each half-sample lexicon, then compute per-feature Pearson correlation and mean absolute difference between the two half-sample feature vectors.

In [ ]:
import numpy as np
from scipy import stats

os.chdir(project_root)

_NOTEBOOK_DIR = project_root / "lexicons" / "lexicons_validation"
SAMPLE_1_PATH = _NOTEBOOK_DIR / "half_samples" / "lexicons_sample_1.json"
SAMPLE_2_PATH = _NOTEBOOK_DIR / "half_samples" / "lexicons_sample_2.json"

ukp_train_path = project_root / "train_test_files" / "ukp_train_raw.csv"
ukp_test_path = project_root / "train_test_files" / "ukp_test_raw.csv"

if ukp_train_path.exists() and ukp_test_path.exists():
    ukp_train = pd.read_csv(ukp_train_path)
    ukp_test = pd.read_csv(ukp_test_path)
    ukp_all = pd.concat([ukp_train, ukp_test], ignore_index=True)
    print(f"UKP dataset: {len(ukp_all)} arguments")
else:
    ukp_all = pd.DataFrame({
        "argument": [
            text1,
            text2,
            test_text,
            "This policy has clear costs, but the long-term benefits outweigh the risks.",
            "I appreciate your concern, although the evidence points in a different direction.",
            "Without immediate action, the problem will become more expensive and harder to solve.",
        ]
    })
    print(
        "UKP CSVs were not found under train_test_files/. "
        f"Using fallback validation corpus with {len(ukp_all)} arguments."
    )

ukp_all.head(2)


In [ ]:
def score_dataset_with_sample(
    df: pd.DataFrame,
    sample_abs_path: str,
    text_col: str = "argument"
) -> tuple:
    """Score all rows using a specific half-sample lexicon. Returns (df_sub, df_mean)."""
    os.environ["PI_LEXICON_FILE"] = str(Path(sample_abs_path).resolve())
    _load_lexicons.cache_clear()
    _PATTERN_CACHE.clear()
    return build_score_matrices_no_lexicon_specific(df, text_col=text_col)

print("Scoring with sample 1 ...")
sub1, mean1 = score_dataset_with_sample(ukp_all, SAMPLE_1_PATH)
print(f"  Done. Shape: {sub1.shape}")

print("Scoring with sample 2 ...")
sub2, mean2 = score_dataset_with_sample(ukp_all, SAMPLE_2_PATH)
print(f"  Done. Shape: {sub2.shape}")


In [ ]:
sub1.shape, sub2.shape, mean1.shape, mean2.shape

In [ ]:
def internal_consistency_table(df1: pd.DataFrame, df2: pd.DataFrame, label: str = "") -> pd.DataFrame:
    """Per-feature Pearson r and MAD between two feature matrices scored on the same texts."""
    rows = []
    for col in df1.columns:
        if col not in df2.columns:
            continue
        v1, v2 = df1[col].values, df2[col].values
        if v1.std() < 1e-10 or v2.std() < 1e-10:
            r, p = np.nan, np.nan
        else:
            r, p = stats.pearsonr(v1, v2)
        rows.append({"feature": col, "pearson_r": r, "p_value": p,
                     "mean_abs_diff": np.mean(np.abs(v1 - v2))})

    result = pd.DataFrame(rows).set_index("feature").sort_values("pearson_r", ascending=True)
    valid_r = result["pearson_r"].dropna()
    print(f"=== {label} ===")
    print(f"  n features: {len(result)} ({valid_r.notna().sum()} with valid r)")
    print(f"  Pearson r  mean={valid_r.mean():.3f}  median={valid_r.median():.3f}  "
          f"min={valid_r.min():.3f}  max={valid_r.max():.3f}")
    print(f"  r < 0.5: {(valid_r < 0.5).sum()}   r ≥ 0.7: {(valid_r >= 0.7).sum()}   "
          f"r ≥ 0.9: {(valid_r >= 0.9).sum()}")
    return result

ic_sub  = internal_consistency_table(sub1,  sub2,  label="Sub-features")
ic_mean = internal_consistency_table(mean1, mean2, label="Category means")

# Drop features not driven by the LLM lexicons (external resources or fixed formulas)
NON_LEXICON_FEATURES = {
    "Sentiment.vader_compound", "Sentiment.valence", "Sentiment.arousal",
    "Sentiment.dominance", "Sentiment.anger", "Sentiment.sadness", "Sentiment.joy_gain",
    "Engagement.past", "Engagement.imagery", "Engagement.self_reference",
    "Engagement.characters", "Engagement.identification", "Engagement.inquiry",
    "Specificity.lexical_concreteness", "Specificity.interactional_immediacy",
    "Style.length", "Style.fluency", "Style.rhetorical_punctuation",
    "Authority/Credibility.titles", "Authority/Credibility.organizations",
    "Authority/Credibility.consensus", "Evidence.named_entities",
}
# Category means where every subfeature is non-lexical
NON_LEXICON_CATEGORIES = {"Engagement.mean", "Style.mean"}

ic_sub  = ic_sub.drop(index=[f for f in NON_LEXICON_FEATURES if f in ic_sub.index])
ic_mean = ic_mean.drop(index=[f for f in NON_LEXICON_CATEGORIES if f in ic_mean.index])

In [ ]:
# Sub-feature internal consistency sorted by Pearson r (worst first)
display(ic_sub.round(4))


In [ ]:
# LaTeX version of sub-feature internal consistency (lexicon-driven features only)
ic_sub_latex = (
    ic_sub
    .drop(index=[f for f in NON_LEXICON_FEATURES if f in ic_sub.index])
    .dropna(subset=["pearson_r"])
    .copy()
)


def fmt_pval(p):
    if p < 0.001:
        return r"$<.001$"
    return f"{p:.3f}"


ic_sub_fmt = ic_sub_latex[["pearson_r", "mean_abs_diff"]].round(3).astype(str)
ic_sub_fmt["p_value"] = ic_sub_latex["p_value"].apply(fmt_pval)
ic_sub_fmt = ic_sub_fmt[["pearson_r", "p_value", "mean_abs_diff"]]

print(ic_sub_fmt.to_latex(
    caption="Sub-feature internal consistency (lexicon-driven features only), "
            "sorted by Pearson $r$ (worst first). Scores computed using "
            "two random half-splits of the expanded lexicon.",
    label="tab:ic_sub",
    column_format="lrrr",
    escape=False,
))


In [ ]:
valid_r = ic_sub["pearson_r"].dropna()
print(ic_sub.describe())
print(f"median r = {valid_r.median():.3f}")
print(f"n features: {len(ic_sub)}")
print(f"r < 0.5: {(valid_r < 0.5).sum()}")

In [ ]:
# Category-mean internal consistency
display(ic_mean.round(4))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df_ic, title in [
    (axes[0], ic_sub,  "Sub-features"),
    (axes[1], ic_mean, "Category means"),
]:
    r_vals = df_ic["pearson_r"].dropna().sort_values()
    colors = ["#d62728" if r < 0.5 else "#ff7f0e" if r < 0.7 else "#2ca02c" for r in r_vals]
    ax.barh(range(len(r_vals)), r_vals.values, color=colors)
    ax.set_yticks(range(len(r_vals)))
    ax.set_yticklabels(r_vals.index, fontsize=8)
    ax.axvline(x=0.7, color="gray", linestyle="--", linewidth=0.8, label="r=0.7")
    ax.axvline(x=0.9, color="black", linestyle="--", linewidth=0.8, label="r=0.9")
    ax.set_xlim(-0.1, 1.05)
    ax.set_xlabel("Pearson r  (sample 1 vs sample 2)")
    ax.set_title(f"Internal Consistency — {title}")
    ax.legend(fontsize=8)

plt.tight_layout()
out_fig = os.path.join(_NOTEBOOK_DIR, "half_samples", "internal_consistency.pdf")
# plt.savefig(out_fig, bbox_inches="tight")
plt.show()
# print(f"Saved: {out_fig}")


In [ ]:
import numpy as np
from scipy import stats

def correlate_feature_vectors_lexicon_only(df1: pd.DataFrame, df2: pd.DataFrame, label: str = ""):
    """
    Computes correlation between global feature means, excluding non-lexical 
    and zero-variance features to show the true stability of the lexicon.
    """
    # 1. Identify common columns
    common_cols = df1.columns.intersection(df2.columns)
    
    # 2. Identify which columns are "Genuinely Lexical"
    # We use a quick check: if the column is identical across both DFs, it's non-lexical
    lexical_cols = []
    for col in common_cols:
        v1, v2 = df1[col].values, df2[col].values
        
        # Calculate correlation for this specific feature
        if v1.std() < 1e-10 or v2.std() < 1e-10:
            continue # Skip zero-variance
            
        r_feature, _ = stats.pearsonr(v1, v2)
        
        # Only include if the feature actually changed (r < 1.0)
        if r_feature < 0.9999:
            lexical_cols.append(col)
    
    # 3. Calculate means only for those lexical columns
    vec1 = df1[lexical_cols].mean()
    vec2 = df2[lexical_cols].mean()
    
    # 4. Compute Pearson correlation between the two mean vectors
    if len(lexical_cols) < 2:
        print(f"=== Global Vector Correlation: {label} ===")
        print("  Not enough lexical features to correlate.")
        return np.nan, np.nan

    r_global, p_global = stats.pearsonr(vec1, vec2)
    
    print(f"=== Global Vector Correlation (Lexicon-Only): {label} ===")
    print(f"  Lexical Features compared: {len(lexical_cols)} (Filtered out {len(common_cols) - len(lexical_cols)} static/NaN)")
    print(f"  Pearson r: {r_global:.4f}")
    print(f"  p-value:   {p_global:.4e}")
    
    return r_global, p_global

# Execute the honest correlation
r_sub_h, p_sub_h = correlate_feature_vectors_lexicon_only(sub1, sub2, label="Sub-features")
r_mean_h, p_mean_h = correlate_feature_vectors_lexicon_only(mean1, mean2, label="Category means")